In [ ]:
"""
Soccerway — Saudi Pro League Master Scraper
===========================================
Scrapes ALL data from us.soccerway.com and saves to ONE master CSV.

Sections scraped:
  • Standings (Overall / Home / Away) — Pos, Team, GP, W, D, L, GF, GA, GD, Pts, Form
  • Match Results                      — Date, Home, Score, Away, Result, Goals
  • Upcoming Fixtures                  — Date, Home Team, Away Team
  • Top Scorers                        — Rank, Player, Team, Goals, Assists, Matches
  • Form Table                         — Recent form per team
  • Over/Under Table                   — Goals stats per team

All sections → soccerway_data/master.csv  (one file, "section" column separates them)
Individual CSVs also saved per section.

INSTALL:
    pip install selenium webdriver-manager beautifulsoup4 lxml pandas colorama

RUN:
    python soccerway_scraper.py

NEEDS: Google Chrome installed on your machine.
"""

import os, re, sys, time
from datetime import datetime

try:
    import pandas as pd
except ImportError:
    sys.exit("pip install pandas")

try:
    from bs4 import BeautifulSoup
except ImportError:
    sys.exit("pip install beautifulsoup4 lxml")

try:
    from colorama import init, Fore, Style
    init(autoreset=True)
except ImportError:
    class Fore:
        GREEN=CYAN=RED=YELLOW=WHITE=RESET=""
    class Style:
        BRIGHT=RESET_ALL=""

try:
    from selenium import webdriver
    from selenium.webdriver.chrome.options import Options
    from selenium.webdriver.chrome.service import Service
    from selenium.webdriver.common.by import By
    from selenium.webdriver.support.ui import WebDriverWait
    from selenium.webdriver.support import expected_conditions as EC
    from webdriver_manager.chrome import ChromeDriverManager
except ImportError:
    sys.exit("pip install selenium webdriver-manager")


# ══════════════════════════════════════════════════════
#  CONFIG  ← change SEASON/TOKEN if Soccerway updates
# ══════════════════════════════════════════════════════
BASE    = "https://us.soccerway.com"
SEASON  = "saudi-professional-league-2024-2025"
TOKEN   = "v5VHH4vs"   # competition token from page links
OUT     = "soccerway_data"
os.makedirs(OUT, exist_ok=True)

NOW = datetime.now().strftime("%Y-%m-%d %H:%M")

PAGES = {
    "standings_overall": f"{BASE}/saudi-arabia/{SEASON}/standings/{TOKEN}/standings/overall/",
    "standings_home":    f"{BASE}/saudi-arabia/{SEASON}/standings/{TOKEN}/standings/home/",
    "standings_away":    f"{BASE}/saudi-arabia/{SEASON}/standings/{TOKEN}/standings/away/",
    "form_table":        f"{BASE}/saudi-arabia/{SEASON}/standings/{TOKEN}/form/",
    "over_under":        f"{BASE}/saudi-arabia/{SEASON}/standings/{TOKEN}/over_under/",
    "top_scorers":       f"{BASE}/saudi-arabia/{SEASON}/standings/{TOKEN}/top-scorers/",
    "results":           f"{BASE}/saudi-arabia/{SEASON}/results/",
    "fixtures":          f"{BASE}/saudi-arabia/{SEASON}/fixtures/",
}

JS_WAIT = 15   # seconds — increase to 20 on slow machines


# ══════════════════════════════════════════════════════
#  DISPLAY
# ══════════════════════════════════════════════════════
def banner():
    print(f"""
{Fore.GREEN}{Style.BRIGHT}╔══════════════════════════════════════════════════════╗
║   ⚽  Soccerway — Saudi Pro League Master Scraper   ║
║      {SEASON}      ║
║      All data → soccerway_data/master.csv           ║
╚══════════════════════════════════════════════════════╝{Style.RESET_ALL}
""")

def section(n, total, title):
    print(f"\n{Fore.CYAN}{Style.BRIGHT}{'─'*58}")
    print(f"  {n}/{total}  ·  {title}")
    print(f"{'─'*58}{Style.RESET_ALL}")

def ok(m):   print(f"  {Fore.GREEN}✓{Style.RESET_ALL}  {m}")
def warn(m): print(f"  {Fore.YELLOW}⚠{Style.RESET_ALL}  {m}")
def err(m):  print(f"  {Fore.RED}✗{Style.RESET_ALL}  {m}")
def info(m): print(f"  {Fore.CYAN}→{Style.RESET_ALL}  {m}")


# ══════════════════════════════════════════════════════
#  SELENIUM
# ══════════════════════════════════════════════════════
def build_driver():
    opts = Options()
    opts.add_argument("--headless=new")
    opts.add_argument("--no-sandbox")
    opts.add_argument("--disable-dev-shm-usage")
    opts.add_argument("--disable-gpu")
    opts.add_argument("--window-size=1920,1080")
    opts.add_argument("--user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
                      "AppleWebKit/537.36 (KHTML, like Gecko) "
                      "Chrome/124.0.0.0 Safari/537.36")
    opts.add_argument("--disable-blink-features=AutomationControlled")
    opts.add_experimental_option("excludeSwitches", ["enable-automation"])
    opts.add_experimental_option("useAutomationExtension", False)
    info("Starting headless Chrome …")
    svc    = Service(ChromeDriverManager().install())
    drv    = webdriver.Chrome(service=svc, options=opts)
    drv.execute_cdp_cmd(
        "Page.addScriptToEvaluateOnNewDocument",
        {"source": "Object.defineProperty(navigator,'webdriver',{get:()=>undefined})"}
    )
    ok("Browser ready.")
    return drv


def fetch(drv, url, wait_css="div.ui-table__row", wait_sec=JS_WAIT):
    info(f"Fetching: {url}")
    drv.get(url)
    try:
        WebDriverWait(drv, wait_sec).until(
            EC.presence_of_element_located((By.CSS_SELECTOR, wait_css))
        )
        ok(f"Page rendered  ({wait_css})")
    except Exception:
        warn(f"Timeout after {wait_sec}s — parsing available HTML.")
    time.sleep(2)
    drv.execute_script("window.scrollTo(0, document.body.scrollHeight * 0.5)")
    time.sleep(1.5)
    return BeautifulSoup(drv.page_source, "lxml")


def click_show_more(drv, times=8):
    """Click 'Show more games' link to load additional match rows."""
    clicked = 0
    for i in range(times):
        try:
            # The link text is exactly "Show more games"
            btns = drv.find_elements(By.PARTIAL_LINK_TEXT, "Show more")
            if not btns:
                break
            btn = btns[-1]   # bottom-of-page "show more games" link
            drv.execute_script("arguments[0].scrollIntoView({block:'center'});", btn)
            time.sleep(0.6)
            drv.execute_script("arguments[0].click();", btn)
            clicked += 1
            time.sleep(2.5)
        except Exception:
            break
    return clicked


# ══════════════════════════════════════════════════════
#  PARSER A — STANDINGS TABLE
#  div.ui-table__row  (Soccerway's custom component)
#  Columns from header: #, Team, GP, W, T(draw), L, G, GD, Pts, Form
# ══════════════════════════════════════════════════════
def parse_standings(soup, section_label):
    rows = soup.find_all("div", class_="ui-table__row")
    if not rows:
        return []

    data = []
    for row in rows:
        # ── Position ──────────────────────────────────
        rank_div = row.find("div", class_="tableCellRank")
        if not rank_div:
            continue
        try:
            pos = int(rank_div.get_text(strip=True).rstrip("."))
        except ValueError:
            continue

        # ── Team ──────────────────────────────────────
        name_a = row.find("a", class_="tableCellParticipant__name")
        if not name_a:
            continue
        team     = name_a.get_text(strip=True)
        href     = (row.find("a", class_="tableCellParticipant__image") or name_a).get("href","")
        team_url = (BASE + href) if href.startswith("/") else href

        # ── Numeric cells ─────────────────────────────
        # plain value spans = P, W, D, L
        all_val = row.find_all("span", class_=lambda c: c and "table__cell--value" in c)
        plain   = [s for s in all_val if not any(
                    x in (s.get("class") or [])
                    for x in ["table__cell--score",
                               "table__cell--goalsForAgainstDiff",
                               "table__cell--points"])]

        def vi(lst, i):
            try:    return int(lst[i].get_text(strip=True))
            except: return 0

        gp  = vi(plain, 0)
        w   = vi(plain, 1)
        d   = vi(plain, 2)   # "T" on Soccerway = Draw
        l   = vi(plain, 3)

        # Goals "GF:GA"
        sc  = row.find("span", class_=lambda c: c and "table__cell--score" in c)
        gf = ga = 0
        if sc:
            m = re.search(r"(\d+)\s*:\s*(\d+)", sc.get_text())
            if m:
                gf, ga = int(m.group(1)), int(m.group(2))

        # GD
        gd_sp = row.find("span", class_=lambda c: c and "goalsForAgainstDiff" in c)
        gd    = gf - ga
        if gd_sp:
            try:    gd = int(gd_sp.get_text(strip=True))
            except: pass

        # Points
        pts_sp = row.find("span", class_=lambda c: c and "table__cell--points" in c)
        pts    = 0
        if pts_sp:
            try:    pts = int(pts_sp.get_text(strip=True))
            except: pass

        # Form badges — data-testid="wcl-badgeForm-win/draw/loss"
        form_div = row.find("div", class_=lambda c: c and "table__cell--form" in c)
        form = ""
        if form_div:
            for badge in form_div.find_all(attrs={"data-testid": True}):
                tid = badge.get("data-testid","").lower()
                form += "W" if "win" in tid else "L" if "loss" in tid or "lose" in tid else "D"
            if not form:
                for sp in form_div.find_all("span"):
                    t = sp.get_text(strip=True).upper()
                    if t in ("W","D","L"):
                        form += t

        # Win % and avg goals
        win_pct      = round(w / gp * 100, 1) if gp else 0
        avg_scored   = round(gf / gp, 2)      if gp else 0
        avg_conceded = round(ga / gp, 2)       if gp else 0
        ppg          = round(pts / gp, 2)      if gp else 0

        data.append({
            "section":        section_label,
            "pos":            pos,
            "team":           team,
            "team_url":       team_url,
            "played":         gp,
            "won":            w,
            "drawn":          d,
            "lost":           l,
            "goals_for":      gf,
            "goals_against":  ga,
            "goal_diff":      gd,
            "points":         pts,
            "ppg":            ppg,
            "win_pct":        win_pct,
            "avg_scored":     avg_scored,
            "avg_conceded":   avg_conceded,
            "form_last5":     form[-5:],
            "scraped_at":     NOW,
        })

    return data


# ══════════════════════════════════════════════════════
#  PARSER B — MATCH RESULTS & FIXTURES
#  Results page uses  div.event__match  rows
#  Each contains: event__time, event__participant--home/away, event__score
# ══════════════════════════════════════════════════════
def parse_matches(soup, kind="result"):
    """
    Parse div.event__match rows.
    Structure:
      div.event__time                                  → date "May 26, 2025"
      div.event__homeParticipant > span.wcl-name_*     → home team name
      div.event__awayParticipant > span.wcl-name_*     → away team name
      span.event__score--home                          → home goals (results only)
      span.event__score--away                          → away goals (results only)
    """
    records = []
    match_rows = soup.find_all("div", class_="event__match")

    for row in match_rows:
        # Date
        date_div = row.find("div", class_=re.compile(r"event__time", re.I))
        date_str = date_div.get_text(strip=True) if date_div else ""

        # Teams — name is in a span with class starting "wcl-name"
        home_div = row.find("div", class_=re.compile(r"event__homeParticipant", re.I))
        away_div = row.find("div", class_=re.compile(r"event__awayParticipant", re.I))
        home_span = home_div.find("span", class_=re.compile(r"^wcl-name|wcl-name_", re.I)) if home_div else None
        away_span = away_div.find("span", class_=re.compile(r"^wcl-name|wcl-name_", re.I)) if away_div else None
        home = home_span.get_text(strip=True) if home_span else (home_div.get_text(strip=True) if home_div else "")
        away = away_span.get_text(strip=True) if away_span else (away_div.get_text(strip=True) if away_div else "")

        if not home or not away or home == away:
            continue

        rec = {
            "section":    kind,
            "date":       date_str,
            "home_team":  home,
            "away_team":  away,
            "scraped_at": NOW,
        }

        # Score (results only)
        home_sc = row.find("span", class_=re.compile(r"event__score--home", re.I))
        away_sc = row.find("span", class_=re.compile(r"event__score--away", re.I))
        if home_sc and away_sc:
            try:
                hg = int(home_sc.get_text(strip=True))
                ag = int(away_sc.get_text(strip=True))
                rec.update({
                    "home_goals":  hg,
                    "away_goals":  ag,
                    "score":       f"{hg}-{ag}",
                    "result":      "H" if hg > ag else "A" if ag > hg else "D",
                    "total_goals": hg + ag,
                })
            except ValueError:
                pass

        records.append(rec)

    return records


# ══════════════════════════════════════════════════════
#  PARSER C — TOP SCORERS
# ══════════════════════════════════════════════════════
def parse_top_scorers(soup):
    rows = soup.find_all("div", class_="ui-table__row")
    data = []
    for row in rows:
        # Rank — try tableCellRank, fall back to any leading number cell
        rd = row.find("div", class_=re.compile(r"tableCellRank|tableCellRanking", re.I))
        rank = None
        if rd:
            txt = rd.get_text(strip=True).rstrip(".")
            if txt.isdigit():
                rank = int(txt)
        if rank is None:
            # fallback: first text node that's a plain number
            first_txt = row.get_text(" ", strip=True).split(" ")[0].rstrip(".")
            if first_txt.isdigit():
                rank = int(first_txt)
        if rank is None:
            continue

        # Player / team names — all participant-name links
        parts = row.find_all("a", class_=re.compile(r"tableCellParticipant__name|participant.*name", re.I))
        if not parts:
            parts = [a for a in row.find_all("a") if a.get_text(strip=True)]
        player = parts[0].get_text(strip=True) if len(parts) > 0 else ""
        team   = parts[1].get_text(strip=True) if len(parts) > 1 else ""

        player_href = parts[0].get("href","") if parts else ""
        player_url  = (BASE + player_href) if player_href.startswith("/") else player_href

        # Numeric stat columns: Goals, Assists, Matches (in order)
        vals = [s.get_text(strip=True)
                for s in row.find_all("span", class_=lambda c: c and "table__cell--value" in c)]
        if not vals:
            # fallback: any span with pure-digit text
            vals = [s.get_text(strip=True) for s in row.find_all("span")
                    if s.get_text(strip=True).isdigit()]

        def vint(i):
            try:    return int(vals[i])
            except: return 0

        if not player:
            continue

        data.append({
            "section":    "top_scorers",
            "rank":       rank,
            "player":     player,
            "player_url": player_url,
            "team":       team,
            "goals":      vint(0),
            "assists":    vint(1),
            "matches":    vint(2),
            "scraped_at": NOW,
        })
    return data


# ══════════════════════════════════════════════════════
#  PARSER D — GENERIC (form table, over/under)
#  Reads column headers dynamically then maps values
# ══════════════════════════════════════════════════════
def parse_generic(soup, section_label):
    # Get column headers
    hdr_div = soup.find("div", class_=re.compile(r"ui-table__header", re.I))
    headers = []
    if hdr_div:
        for cell in hdr_div.find_all("div", class_=re.compile(r"headerCell", re.I)):
            h = cell.get_text(strip=True)
            if h:
                headers.append(h)

    rows = soup.find_all("div", class_="ui-table__row")
    data = []
    for row in rows:
        rd = row.find("div", class_="tableCellRank")
        if not rd:
            continue
        try:
            pos = int(rd.get_text(strip=True).rstrip("."))
        except ValueError:
            continue

        name_a = row.find("a", class_="tableCellParticipant__name")
        team   = name_a.get_text(strip=True) if name_a else ""

        vals = [s.get_text(strip=True)
                for s in row.find_all("span", class_=lambda c: c and "table__cell--value" in c)]

        rec = {"section": section_label, "pos": pos, "team": team}
        # Map values to header names (skip first 2 headers: # and Team)
        col_names = headers[2:] if len(headers) > 2 else []
        for i, v in enumerate(vals):
            col = col_names[i].replace(" ","_").lower() if i < len(col_names) else f"col_{i}"
            rec[col] = v

        rec["scraped_at"] = NOW
        data.append(rec)
    return data


# ══════════════════════════════════════════════════════
#  PRINT HELPERS
# ══════════════════════════════════════════════════════
def print_standings_table(data):
    print(f"\n  {'#':>3}  {'Team':<24} {'GP':>3} {'W':>3} {'D':>3} {'L':>3}"
          f" {'GF':>4} {'GA':>4} {'GD':>5} {'Pts':>4}  {'Form':<6}  ppg  W%")
    print(f"  {'─'*82}")
    for r in data:
        gd = f"+{r['goal_diff']}" if r['goal_diff'] > 0 else str(r['goal_diff'])
        print(f"  {r['pos']:>3}. {r['team']:<24}"
              f" {r['played']:>3} {r['won']:>3} {r['drawn']:>3} {r['lost']:>3}"
              f" {r['goals_for']:>4} {r['goals_against']:>4} {gd:>5}"
              f" {r['points']:>4}  {r['form_last5']:<6}  {r['ppg']:.2f} {r['win_pct']}%")


def print_matches(data, label, n=20):
    if not data:
        return
    print(f"\n  {'Date':<14} {'Home Team':<24} {'Score':>6}  {'Away Team':<24} {'Result'}")
    print(f"  {'─'*80}")
    for r in data[:n]:
        sc = r.get("score", "vs")
        print(f"  {str(r.get('date','')):<14} {r['home_team']:<24}"
              f" {sc:>6}  {r['away_team']:<24} {r.get('result','')}")
    if len(data) > n:
        print(f"  … +{len(data)-n} more rows in master.csv")


def print_scorers(data):
    if not data:
        return
    print(f"\n  {'#':>3}  {'Player':<26} {'Team':<22} {'G':>4} {'A':>4} {'GP':>4}")
    print(f"  {'─'*68}")
    for r in data[:20]:
        print(f"  {r['rank']:>3}. {r['player']:<26} {r['team']:<22}"
              f" {r['goals']:>4} {r['assists']:>4} {r['matches']:>4}")


# ══════════════════════════════════════════════════════
#  SAVE CSV  (individual + append to master)
# ══════════════════════════════════════════════════════
def save(data: list[dict], filename: str, all_frames: list):
    if not data:
        warn(f"No data — skipping {filename}")
        return
    df   = pd.DataFrame(data)
    path = os.path.join(OUT, filename)
    df.to_csv(path, index=False, encoding="utf-8-sig")
    ok(f"Saved {filename}  ({len(df)} rows)")
    all_frames.append(df)


# ══════════════════════════════════════════════════════
#  MAIN
# ══════════════════════════════════════════════════════
def main():
    banner()
    drv = build_driver()
    all_frames = []   # collect all dataframes for master CSV

    try:
        # ── 1. STANDINGS OVERALL ──────────────────────
        section(1, 8, "STANDINGS — OVERALL")
        soup = fetch(drv, PAGES["standings_overall"])
        data = parse_standings(soup, "standings_overall")
        save(data, "standings_overall.csv", all_frames)
        print_standings_table(data)

        # ── 2. STANDINGS HOME ─────────────────────────
        section(2, 8, "STANDINGS — HOME")
        soup = fetch(drv, PAGES["standings_home"])
        data = parse_standings(soup, "standings_home")
        save(data, "standings_home.csv", all_frames)
        print_standings_table(data)

        # ── 3. STANDINGS AWAY ─────────────────────────
        section(3, 8, "STANDINGS — AWAY")
        soup = fetch(drv, PAGES["standings_away"])
        data = parse_standings(soup, "standings_away")
        save(data, "standings_away.csv", all_frames)
        print_standings_table(data)

        # ── 4. MATCH RESULTS ──────────────────────────
        section(4, 8, "MATCH RESULTS")
        soup = fetch(drv, PAGES["results"],
                     wait_css="div.event__match")
        n_clicked = click_show_more(drv, times=10)
        if n_clicked:
            info(f"Clicked 'Show more games' {n_clicked}x to load full season.")
            time.sleep(1)
        soup = BeautifulSoup(drv.page_source, "lxml")
        data = parse_matches(soup, "result")
        # Deduplicate (Show more can re-render existing rows)
        seen, dedup = set(), []
        for r in data:
            key = (r["date"], r["home_team"], r["away_team"], r.get("score",""))
            if key not in seen:
                seen.add(key)
                dedup.append(r)
        data = dedup
        save(data, "results.csv", all_frames)
        print_matches(data, "Results", n=15)

        # ── 5. FIXTURES ───────────────────────────────
        section(5, 8, "UPCOMING FIXTURES")
        soup = fetch(drv, PAGES["fixtures"],
                     wait_css="div.event__match")
        n_clicked = click_show_more(drv, times=8)
        if n_clicked:
            info(f"Clicked 'Show more games' {n_clicked}x to load more fixtures.")
            time.sleep(1)
        soup = BeautifulSoup(drv.page_source, "lxml")
        data = parse_matches(soup, "fixture")
        seen, dedup = set(), []
        for r in data:
            key = (r["date"], r["home_team"], r["away_team"])
            if key not in seen:
                seen.add(key)
                dedup.append(r)
        data = dedup
        save(data, "fixtures.csv", all_frames)
        print_matches(data, "Fixtures", n=15)

        # ── 6. TOP SCORERS ────────────────────────────
        section(6, 8, "TOP SCORERS")
        soup = fetch(drv, PAGES["top_scorers"])
        data = parse_top_scorers(soup)
        save(data, "top_scorers.csv", all_frames)
        print_scorers(data)

        # ── 7. FORM TABLE ─────────────────────────────
        section(7, 8, "FORM TABLE")
        soup = fetch(drv, PAGES["form_table"])
        data = parse_generic(soup, "form_table")
        save(data, "form_table.csv", all_frames)
        for r in data[:5]:
            print(f"  {r}")

        # ── 8. OVER/UNDER ─────────────────────────────
        section(8, 8, "OVER / UNDER GOALS TABLE")
        soup = fetch(drv, PAGES["over_under"])
        data = parse_generic(soup, "over_under")
        save(data, "over_under.csv", all_frames)
        for r in data[:5]:
            print(f"  {r}")

    finally:
        drv.quit()
        ok("Browser closed.")

    # ── MERGE INTO MASTER CSV ─────────────────────────
    if all_frames:
        master = pd.concat(all_frames, ignore_index=True, sort=False)
        master_path = os.path.join(OUT, "master.csv")
        master.to_csv(master_path, index=False, encoding="utf-8-sig")
        print(f"\n{Fore.GREEN}{Style.BRIGHT}")
        print(f"  ╔══════════════════════════════════════════════╗")
        print(f"  ║  MASTER CSV SAVED                           ║")
        print(f"  ║  {master_path:<44}║")
        print(f"  ║  {len(master):,} total rows · {len(master.columns)} columns           ║")
        print(f"  ╚══════════════════════════════════════════════╝")
        print(f"{Style.RESET_ALL}")
        print(f"  Columns in master.csv:")
        for col in master.columns:
            print(f"    • {col}")
    else:
        err("Nothing scraped — master.csv not created.")

    # ── FILE SUMMARY ──────────────────────────────────
    print(f"\n  {'File':<35} {'Rows':>6}")
    print(f"  {'─'*44}")
    for fname in ["standings_overall.csv","standings_home.csv","standings_away.csv",
                  "results.csv","fixtures.csv","top_scorers.csv",
                  "form_table.csv","over_under.csv","master.csv"]:
        path = os.path.join(OUT, fname)
        if os.path.exists(path):
            n = sum(1 for _ in open(path, encoding="utf-8-sig")) - 1
            print(f"  {Fore.GREEN}✓{Style.RESET_ALL}  {fname:<33} {n:>6}")
        else:
            print(f"  {Fore.RED}✗{Style.RESET_ALL}  {fname:<33} {'—':>6}")

    print(f"\n  All files saved to: {Fore.CYAN}./{OUT}/{Style.RESET_ALL}\n")


if __name__ == "__main__":
    main()
